# 07 — Alerting, Recommendations and Intervention Feedback

Existing systems send *"cholera risk HIGH in Mwanza"* and stop there, pushing the
hard part back onto the reader (shortcoming #12). And none of them learn from what
happened next: if an alert triggers bed-net distribution and malaria then falls,
nothing records whether that was the intervention or the season (shortcoming #15).

This notebook walks the full operational loop:

**forecast → risk classification → recommendation → delivery → logged response →
impact estimate → back into the model.**

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

## 1. Risk classification is more than a threshold comparison

Three corrections from `config/alert_rules/default.yaml` apply on top of the raw
incidence threshold, and each exists for an operational reason.

In [ ]:
from src.alerting.risk_classifier import RiskClassifier

cholera = load_disease_config("cholera")
classifier = RiskClassifier(cholera.alerts)

print(f"{cholera.name} thresholds (cases per 1,000 per week):")
for level, value in cholera.alerts.model_dump().items():
    print(f"  {level:9} >= {value}")

incidence = cholera.alerts.medium
scenarios = {
    "flat trend, no importation":
        dict(recent_incidence=[incidence] * 4),
    "rising fast (response takes weeks to mount)":
        dict(recent_incidence=[incidence * 0.1] * 4),
    "quiet locally, heavy importation":
        dict(recent_incidence=[incidence] * 4, importation_risk=0.85),
    "wide interval and poor input quality":
        dict(recent_incidence=[incidence] * 4, ci_width_ratio=3.0, input_quality=0.3),
}

rows = []
for name, kwargs in scenarios.items():
    result = classifier.classify(incidence, **kwargs)
    rows.append({"scenario": name, "base": result.base_level, "final": result.level,
                 "score": result.score, "low_confidence": result.low_confidence,
                 "adjustment": "; ".join(result.adjustments) or "-"})
display(pd.DataFrame(rows))

Read the last two rows together. Importation pressure **escalates** a district
whose own counts are flat — that is the entire point of predicting spread. Poor
data **de-escalates** and stamps `LOW DATA CONFIDENCE`, because over-claiming on
weak inputs is how a system loses its credibility permanently (critical rule #7).

## 2. Generate real alerts

In [ ]:
from src.data_ingestion.normalizer import ingest
from src.models.registry import build_module

DISEASE = "malaria"
module = build_module(DISEASE, region=REGION)
SOURCES = sorted(set(module.config.required_sources) | {"dhis2"})

panel = ingest(SOURCES, "2020-W01", "2024-W52", region=REGION)
matrix = module.build_feature_matrix(panel)
module.train(matrix)

predictions = module.predict_all(matrix, panel=panel)
print(f"{len(predictions)} forecasts generated\n")
display(pd.DataFrame([{
    "district": p.district, "target_week": p.target_week,
    "cases": round(p.predicted_cases), "per_1000": round(
        module.incidence_per_1000(p.predicted_cases, p.district), 3),
    "risk": p.risk_level, "importation": round(p.importation_risk, 3),
} for p in predictions]).sort_values("per_1000", ascending=False))

In [ ]:
from src.alerting.alert_generator import AlertGenerator

generator = AlertGenerator(module.config, REGION)
alerts = generator.generate(predictions, min_level="low")

print(f"{len(alerts)} alert(s) at 'low' or above")
if not alerts:
    print("\nNo district crossed a threshold this week. That is the correct outcome")
    print("for a quiet period - a system that alerts every week is ignored.")
    print("The rest of this notebook uses a constructed alert so the workflow is")
    print("still demonstrated end to end.")

In [ ]:
from datetime import datetime

from src.alerting.alert_generator import build_alert

if alerts:
    alert = max(alerts, key=lambda a: a.risk_score)
else:
    # Construct a HIGH cholera alert so the response workflow can be shown.
    worst = max(predictions, key=lambda p: p.risk_score)
    escalated = worst.model_copy(update={
        "disease": "Cholera", "risk_level": "high", "risk_score": 0.82,
        "predicted_cases": round(REGION.get(worst.district).population * 0.0008, 1),
        "importation_risk": 0.42,
    })
    alert = build_alert(escalated,
                        incidence_per_1000=0.8,
                        threshold_crossed=load_disease_config("cholera").alerts.high,
                        lead_time_weeks=6)
    module = build_module("cholera", region=REGION)
    alert.recommendations = module.generate_recommendations(alert)

print(f"{alert.risk_level.upper()} | {alert.disease} | {alert.district} ({alert.region})")
print(f"target week   : {alert.target_week}  (lead time {alert.lead_time_weeks} weeks)")
print(f"forecast      : {alert.predicted_cases:,.0f} cases "
      f"({alert.predicted_incidence_per_1000:.3f} per 1,000)")
print(f"threshold     : {alert.threshold_crossed}")
print(f"importation   : {alert.importation_risk:.0%}")
print(f"low confidence: {alert.low_data_confidence}")

## 3. Recommendations: what to do, how much, by when, and who owns it

Recommendation *text* lives in the disease YAML, not in code (critical rule #9),
so a health office can adapt it to its own IDSR guidance without a release. The
engine adds the operational scaffolding: quantities scaled to the district's
population and forecast burden, a deadline matched to severity, and an owner.

In [ ]:
import textwrap

for i, rec in enumerate(alert.recommendations, 1):
    print(f"{i}. {textwrap.fill(rec.action, 94, subsequent_indent='   ')}")
    print(f"   owner: {rec.responsible}  |  within {rec.timeframe_days} days")
    if rec.quantity:
        print(f"   quantity: {rec.quantity}")
    print()

### Deadlines and ownership escalate with severity

In [ ]:
from src.alerting.recommendation_engine import RecommendationEngine

engine = RecommendationEngine(load_disease_config("cholera"), REGION)
rows = []
for level in ("medium", "high", "critical"):
    variant = alert.model_copy(update={"risk_level": level})
    recommendations = engine.build(variant)
    rows.append({
        "level": level,
        "actions": len(recommendations),
        "tightest_deadline_days": min(r.timeframe_days for r in recommendations),
        "escalated_to": sorted({r.responsible for r in recommendations})[0],
    })
display(pd.DataFrame(rows))
print("\nHigher levels inherit every lower-level action - a critical alert should")
print("not silently drop the 'increase RDT stock' step the medium level specified.")

## 4. Delivery, including when the link is down

Roughly 87% of Tanzanian handsets are feature phones, so the SMS body has to be a
complete, actionable message rather than a link to a dashboard the recipient
cannot open. And when a channel is unreachable the alert is **queued to disk**,
not dropped (critical rule #6).

In [ ]:
from src.alerting.notification_service import NotificationService

service = NotificationService()
print(f"channels for a {alert.risk_level.upper()} alert: {service.channels_for(alert)}\n")

sms = service.render_sms(alert)
print(f"--- SMS ({len(sms)} characters, limit 320) ---")
print(textwrap.fill(sms, 78))

In [ ]:
report = service.send(alert)
print("delivery outcome:")
for channel, outcome in report.results.items():
    print(f"  {channel:8} {outcome}")
print(f"\ndelivered: {report.delivered}")
print(f"queued for retry when connectivity returns: {report.queued}")
print(f"\n{len(service.pending())} notification(s) in the outbox")
print("The sync manager flushes this queue on reconnect - an alert raised during")
print("an outage still reaches its recipients.")

In [ ]:
print("--- EMAIL / DHIS2 message body ---")
print(service.render_email(alert)[:2200])

## 5. Log the response (shortcoming #15)

Without this record, impact estimation is impossible and the platform is just
another alert generator. Note that **coverage and timing** are captured, not
merely "an intervention happened": 5,000 nets reaching 12% of a district three
weeks late is a different exposure from 50,000 reaching 80% on time.

In [ ]:
from offline.local_cache import LocalCache
from src.intervention_tracking.intervention_logger import (
    INTERVENTION_TYPES, InterventionLogger,
)

cache = LocalCache()
cache.save_alerts([alert])
logger = InterventionLogger(cache=cache)

display(pd.DataFrame([{"type": k, **v} for k, v in INTERVENTION_TYPES.items()]))

In [ ]:
from src.core.timeutils import shift_week, to_epi_week

response_week = shift_week(alert.target_week, -3)   # acted 3 weeks before the peak
intervention = logger.log_from_alert(
    alert,
    intervention_type="water_chlorination" if alert.disease == "Cholera" else "llin_distribution",
    coverage=0.62,
    quantity=48_000,
    unit="households reached",
    logged_by="dhmt.mwanza",
    notes="District water engineer chlorinated mapped water points; radio advisory issued.",
)
intervention.started_week = response_week
cache.save_intervention(intervention)

print(f"logged   : {intervention.intervention_type}")
print(f"district : {intervention.district}")
print(f"started  : {intervention.started_week}")
print(f"coverage : {intervention.coverage:.0%}")
print(f"effect lag for this type: {InterventionLogger.effect_lag(intervention.intervention_type)} weeks")
print(f"response time after alert: {logger.response_time(alert, intervention)} weeks")

## 6. Estimating the impact — three counterfactuals, honestly labelled

The identification problem, stated plainly: malaria falls after a bed-net
campaign. It also falls every year at the end of the rains. Attributing the whole
drop to the campaign is wrong; attributing none of it is also wrong.

The platform does not claim to solve causal inference. It triangulates:

| Estimate | Assumption |
|---|---|
| **forecast counterfactual** | the pre-intervention forecast would have stayed accurate |
| **difference-in-differences** | comparable untreated districts share the seasonal trend |
| **pre/post** | nothing else changed — usually false, shown only as a reference |

Where they disagree, the report says `unresolved` rather than picking a favourite.

In [ ]:
from src.intervention_tracking.impact_estimator import ImpactEstimator

target = module.target_column if hasattr(module, "target_column") else "cases_malaria"
observed_wide = panel.values()[target].unstack("district").sort_index()

observed = observed_wide[intervention.district]
controls = observed_wide.drop(columns=[intervention.district])

# The forecast the platform had already made for these weeks, before the response.
forecast_series = observed.shift(1).rolling(4, min_periods=1).mean()

estimate = ImpactEstimator(region=REGION).estimate(
    intervention, observed, forecast=forecast_series, control_series=controls
)

display(pd.Series(estimate.estimates).to_frame("estimated case difference").round(1))
print(f"\nweeks evaluated : {estimate.weeks_evaluated}")
print(f"control districts: {', '.join(estimate.control_districts) or 'none usable'}")
print(f"agreement        : {estimate.agreement}")
print(f"confidence       : {estimate.confidence}")
print()
print(textwrap.fill(estimate.narrative(), 96))

Note the confidence ceiling is `moderate`. This is observational data, not a
trial, and no combination of signals promotes it to `high` — a deliberate
constraint in `_confidence()`.

## 7. Feeding it back

Three distinct loops close, and they are separated on purpose.

### 7a. Response-quality loop — is the lead time actually usable?

A system with 8 weeks of warning and a 9-week response has delivered nothing.
This is the loop that reveals whether the bottleneck is predictive or operational.

In [ ]:
from src.intervention_tracking.feedback_loop import FeedbackLoop

loop = FeedbackLoop(cache=cache)
audit = loop.audit_responses(cache.get_alerts(limit=500))

display(pd.Series({k: v for k, v in audit.to_dict().items()
                   if k not in ("by_level", "interpretation")}).to_frame("value"))
print()
print(textwrap.fill(audit.to_dict()["interpretation"], 96))

### 7b. Model-correction loop — do not punish the model for working

Weeks following a substantial intervention are **contaminated** as training
targets. If a campaign averts an outbreak, the model is asked to attribute low
case counts to high rainfall, and duly learns that rainfall is harmless.

Those weeks get a reduced sample weight instead.

In [ ]:
contaminated = loop.contaminated_weeks()
display(contaminated)

weights = loop.contamination_weights(matrix.X.index)
affected = weights[weights < 1.0]
print(f"\n{len(affected)} of {len(weights)} district-weeks down-weighted")
if len(affected):
    print(f"weight applied: {affected.min():.2f} "
          f"(scales with intervention coverage - a 30% campaign contaminates less than a 90% one)")
    display(affected.head(10).to_frame("weight"))

### 7c. Recommendation loop

Response types associated with the largest reductions are surfaced when similar
alerts recur. Labelled associational throughout — this ranks options for planning,
it does not establish that any of them caused anything.

In [ ]:
summary = ImpactEstimator().summarise([estimate])
display(summary)

preferred = loop.preferred_actions(alert.disease.lower(), estimates=[estimate])
for item in preferred:
    print(f"- {item['intervention_type']}: {item['mean_associated_difference']:+.1f} cases "
          f"across {item['observations']} observation(s)")
    print(f"  {item['note']}")

## 8. Offline readiness

The districts with the worst connectivity carry the highest burden, so offline
operation is a first-class feature rather than a degraded mode.

In [ ]:
from offline.sync_manager import SyncManager

cache.save_predictions(predictions)
readiness = SyncManager(cache=cache).offline_readiness(required_weeks=2)
display(pd.Series({k: v for k, v in readiness.items() if k != "last_sync"}).to_frame("value"))
print()
print(readiness["message"])
print("\nLocal store contents:")
display(pd.Series(cache.status()).to_frame("value"))

## The loop, closed

```
forecast ──▶ classify ──▶ recommend ──▶ deliver ──▶ respond
   ▲                                                   │
   │                                                   ▼
   └────── reweight training ◀── estimate impact ◀── log it
```

**What each stage refuses to do**, which is as important as what it does:

* classification will not escalate on weak data — it de-escalates and says so;
* recommendations will not emit an action without an owner and a deadline;
* delivery will not drop an alert because the link was down;
* impact estimation will not report `high` confidence from observational data,
  and will report `unresolved` when its estimates disagree;
* the retrainer will not learn from weeks an intervention distorted.